# Week 3: Key-Value Stores (Redis)

# Introduction:

### Redis:
   <a href="https://redis.io/">Redis</a> Redis is an open source (BSD licensed), in-memory data structure store, used as a database, cache and message broker. It supports data structures such as strings, hashes, lists, sets, sorted sets with range queries, bitmaps, hyperloglogs, geospatial indexes with radius queries and streams.<br/>
   
<img src="https://upload.wikimedia.org/wikipedia/fr/thumb/6/6b/Redis_Logo.svg/640px-Redis_Logo.svg.png" width ="250" >


#### <a href='https://redislabs.com/redis-enterprise/data-structures/'>Redis Data Structures</a>
* Redis is not a plain key-value store, it is actually a data structures server, supporting different kinds of values.
* An introduction to Redis data types and abstractions https://redis.io/topics/data-types-intro
* Redis keys are always strings.


<img src='https://redis.io/wp-content/uploads/2021/07/key-value-data-stores-2-v2-920x612-1.png' width='500' >

### How To Query Redis!

- Commands for each data type for common access patterns, with bulk operations, and partial transaction support.

### PreLab

#### 1. Install Redis on Windows
- Redis is a cross-platform DB, We can install it on Linux, or Windows, ..etc.
- There are two ways to install Redis under Windows
    - Download the latest Redis .msi file from https://github.com/MSOpenTech/redis/r... and install it. 
    
    - You can choose either from these sources
        - https://github.com/microsoftarchive/redis/releases or
        - https://github.com/rgl/redis/downloads

- Personally I prepared the first option
- Download Redis-x64-2.8.2104.zip
- Extract the zip to the prepared directory
- Run redis-server.exe
- Run redis-cli.exe
- For more info follow this setup-video tutorial (https://www.youtube.com/watch?v=188Fy-oCw4w)


#### Linux and Debian 

- Even quicker and dirtier instructions for Debian-based Linux distributions are as follows:
    - download Redis from http://redis.io/download 
    - extract, run make && sudo make install
    - Then run sudo python -m easy_install redis hiredis (hiredis is an optional performance-improving C library).

#### 2. Install the Python Package ("<a href='https://pypi.org/project/redis/'>redis</a>") to connecto to Redis 
- use th command ```pip install redis``` in your command line.


#### (more) Accessing Redis from Command Line:
- Add the Redis installation "/home" and "/bin" directories to the environment variables.
- start Redis server in one command window(CMD, poweshell, ..etc)using the command ```redis-server```.
- In another command window, start your Redis Client using the command ```redis-cli```
- Now you have the Redis Client Shell connected to the default <b>db0</b> DB. 

In [1]:
import redis
from pprint import pprint
import pandas as pd
from time import sleep

import warnings
warnings.filterwarnings('ignore')

##### Get a client connection to redis server, using the url and the port, and Db

In [2]:
r = redis.Redis(host='localhost', port=6379, db=0)

## Task 1: PUB/SUB in REDIS

- <b>"Pub/Sub"</b> aka Publish-Subscribe pattern is a pattern in which there are three main components, **sender**, **receiver** & **broker**.
- It is mainly characterized by **listeners subscribing to channels**, with **publishers** sending **binary string** messages to channels.
- The communication is processed by the broker, it helps the sender or publisher to publish information and deliver that information to the receiver or subscriber.


#### Example of Consumer

- The following Consumer subscribes to 'Two' Channels:
    - (Tartu), and  
    - To any channel that starts with the pattern 'DataEng_Students:'
- If the Producer publishes something related to these channels, this will be listened to and sent to this consumer.

In [3]:
# inspired by: https://gist.github.com/jobliz/2596594

import threading

class Listener(threading.Thread):
    def __init__(self, r):
        threading.Thread.__init__(self)
        self.redis = r
        self.pubsub = self.redis.pubsub()
        
        #Subscribe to these channels
        
        #YOUR CODE LINE HERE to SUBSCRIBE to 'tartuuniv' Channel
        self.pubsub.subscribe("tartuuniv")
        #YOUR CODE LINE HERE to SUBSCRIBE to channels start with the pattern'destudents:'
        self.pubsub.psubscribe("destudents:*")


    def work(self, item):
        print (item['channel'], ":", item['data'])

    def run(self):
        for item in self.pubsub.listen():
            if item['data'] == b"KILL":
                self.pubsub.unsubscribe()
                print (self, "unsubscribed and finished")
                break
            else:
                self.work(item)

if __name__ == '__main__':
    r = redis.Redis('localhost')
    client = Listener(r)
    client.start()

b'tartuuniv' : 1
b'destudents:*' : 2


b'tartuuniv' : b'new_course_created DataEngineering'
b'destudents:1415' : b'John Doe'
<Listener(Thread-4, started 127223334839872)> unsubscribed and finished


#### Start Publishing Some data/messages to different Channels
- In a new command window, run the following commands:
    - (# opens a repl, all subsequent commands should show something in the first terminal)
```
redis-cli 
> publish tartuuniv "new_course_created DataEngineering" 
> publish tartuuniv "new_job_posted Senior_Researcher"
> publish destudents:1415 "John Doe"
> publish destudents:jane "Jane Kidman"
> publish Tallin "nobody listens"
> publish tartuuniv KILL   # this should terminate (listener.py)
```

##### Wrtie what you could notice from the previous Example:

In [ ]:
#YOur Answer Here!!

##### Another Example of Redis Pub/SUB 

Let me explain it with an example. Let’s assume, Joe is the owner of a Music Shop where he sells music of different Genres. Alice is a Musician who publishes/sells her music at Joe’s Shop. And, Bob is Joe’s Customer who buys music from Joe. Joe keeps a list of his customers and their interests, hence he knows Bob likes Classical Music. Whenever Alice composes a Classical Music album in Joe’s shop, Joe delivers it to Bob. The interesting part is Bob doesn’t necessarily have to know who created the Music and Alice also doesn’t have to know who listens to her music. 

In [13]:
import redis

r = redis.Redis('localhost')
p = r.pubsub()
p.subscribe("classic")

while True:
    message=p.get_message()
    if message and not message['data']==1:
        #You can do any kind of computations on the coming/readed messages !!
        message=message['data'].decode('utf-8')
        song,singer=message.split(':')
        #here we just split the message and read it as song,and singer, but you can do any computations
        print("SONG: ", song)
        print("SINGER: ", singer)

SONG:  fre
SINGER:  dez


KeyboardInterrupt: 

## Task3: Hats-Shop Website Scenario: Example

It’s time to break out a fuller example. Let’s pretend we’ve decided to start a lucrative website that sells hats, and hired you to build the site.

* You’ll use Redis to handle some of the product catalog, inventory, and bot traffic detection for our website.
* It’s day one for the site, and we’re going to be selling **three** limited-edition hats. 
* Each hat gets held in a Redis hash of field-value pairs, and the hash has a key that is a prefixed random integer , such as **hat:56854717**. 
* Using the **hat:prefix** is Redis convention for creating a sort of **namespace** within a Redis database:

In [57]:
import random

#we use random to get random prefixes
random.seed(444)
hats = {f"hat:{random.getrandbits(32)}": i for i in (
    {
        "color": "black",
        "price": 49.99,
        "style": "fitted",
        "quantity": 1000,
        "npurchased": 0,
    },
    {
        "color": "maroon",
        "price": 59.99,
        "style": "hipster",
        "quantity": 500,
        "npurchased": 0,
    },
    {
        "color": "green",
        "price": 99.99,
        "style": "baseball",
        "quantity": 200,
        "npurchased": 0,
    })
}

#### Writing Hats data to Redis & Pipelining
- To do an initial write of this data into Redis, we can use .hmset() (hash multi-set), calling it for each dictionary.
- The code block above also introduces the concept of Redis pipelining, which is a way to cut down the number of round-trip transactions that you need to write or read data from your Redis server.

In [58]:
with r.pipeline() as pipe:
    for h_id, hat in hats.items():
        pipe.hmset(h_id, hat)      
    print(pipe.execute())

[True, True, True]


- With a pipeline, all the commands are buffered on the client side and then sent at once, in one fell swoop, using pipe.hmset() in Line 3.
- This is why the three True responses are all returned at once, when you call pipe.execute() in Line 4.

#### Check Existence of Specific Hat with key('hat:56854717')

In [17]:
r.exists("hat:56854717")

1

### Query by Hat id (GetAll fields)
- Get all fields of the hat ('hat:56854717')

In [18]:
r.hgetall("hat:56854717")

{b'color': b'green',
 b'price': b'99.99',
 b'style': b'baseball',
 b'quantity': b'200',
 b'npurchased': b'0'}

### Query by Hat id (Get Specific Fields)
- Get the color, style, and price of ('hat:56854717')

In [20]:
r.hmget("hat:56854717", "color", "style", "price")

[b'green', b'baseball', b'99.99']

#### Get all the Hats in your DB
- Hint: use the pattern '**hat***' as the parmater of **r.keys()** function to specify only the hat keys.

In [21]:
r.keys("hat*")

[b'hat:1326692461', b'hat:56854717', b'hat:1236154736']

#### Insert one more Item(Hash) in the "Hats" HashSet
- {"hat:random Prefix As Seen before, 
   "color": "black",
   "price": 60.99,
   "style": "fedora",
   "quantity": 50,
   "npurchased": 0}

In [56]:
r.hmset(f"hat:{random.getrandbits(32)}",{
        "color": "black",
        "price": 60.99,
        "style": "fedora",
        "quantity": 50,
        "npurchased": 0
    })

True

#### Check if the new item ("Hat") is added !
- You can get all the hats again!

In [27]:
r.keys("hat:*")

[b'hat:1326692461', b'hat:1327727452', b'hat:56854717', b'hat:1236154736']

####  Get the count of Hats in your DB 

In [33]:
print(len(r.keys("hat*")))


4


#### Filter out Hats with prices less than 60

In [37]:
with r.pipeline() as pipe:
    for key in r.keys("hat:*"):
        pipe.hgetall(key)
    results = pipe.execute()

for key, hat_data in zip(r.keys("hat:*"), results):
    # Convert values from bytes to str/int if needed
    hat = {k.decode(): v.decode() for k, v in hat_data.items()}
    if float(hat.get("price", 0)) >= 60:
        print(hat)
    

{'color': 'black', 'price': '60.99', 'style': 'fedora', 'quantity': '50', 'npurchased': '0'}
{'color': 'green', 'price': '99.99', 'style': 'baseball', 'quantity': '200', 'npurchased': '0'}


### Transactions and Keeping Atomicity in Redis

Redis allows the execution of a group of commands in a single step, with two important guarantees:
* All the commands in a transaction are serialized and executed sequentially. It can never happen that a request issued by another client is served in the middle of the execution of a Redis transaction. This guarantees that the commands are executed as a single isolated operation.

* Either all of the commands or none are processed, so a Redis transaction is also atomic.

In redis-py, Pipeline is a transactional pipeline class by default. This means that, even though the class is actually named for something else (pipelining), it can be used to create a transaction block also.

In Redis, a transaction starts with <b> MULTI </b> and ends with <b>EXEC<b>:
    * Everything in between is executed as one all-or-nothing buffered sequence of commands.
    * Methods that you call on pipe effectively buffer all of the commands into one, and then send them to the server in a single request
    
 

In [ ]:
def buyitem(r: redis.Redis, itemid: int) -> None:
    with r.pipeline() as pipe:
        error_count = 0
        while True:
            try:
                # Get available inventory, watching for changes
                # related to this itemid before the transaction
                pipe.watch(itemid)

                nleft: bytes = r.hget(itemid, "quantity")
                if nleft > b"0":
                    pipe.multi()
                    pipe.hincrby(itemid, "quantity", -1)
                    pipe.hincrby(itemid, "npurchased", 1)
                    pipe.execute()
                    break
                else:
                    # Stop watching the itemid and raise to break out
                    pipe.unwatch()
                    raise OutOfStockError(f"Sorry, {itemid} is out of stock!")

            except redis.WatchError:
                # Log total num. of errors by this user to buy this item,
                # then try the same process again for WATCH/HGET/MULTI/EXEC

                error_count += 1

                logging.warning("WatchError #%d: %s; retrying",error_count, itemid)

        return None

#### Call the function buyitem, and buy three hats of the hat 'hat:56854717'

In [43]:
buyitem(r, "hat:56854717")
buyitem(r, "hat:56854717")
buyitem(r, "hat:56854717")
buyitem(r, "hat:56854717")

#### Check the quantity and npurchased feilds of the Hat hash ('hat:56854717')

In [44]:
r.hmget("hat:56854717", "quantity", "npurchased")

[b'192', b'8']

Now, when some poor user is late to the game, they should be met with an <b>"OutOfStockError"</b> that tells our application to render an error message page on the frontend
- Buy remaining 196 hats for item hat:56854717, the stock will be 0!!

In [45]:
for _ in range(196):
    buyitem(r, "hat:56854717")

r.hmget("hat:56854717", "quantity", "npurchased")

NameError: name 'OutOfStockError' is not defined

#### Write down what will happen when you try to buy one more hat of the same key.

In [47]:
try:
    buyitem(r, "hat:56854717")
except:
    print("OutofStockError")

OutofStockError


<font color='red'>Answer:</font>

#### Delete elements (delete hats with 'black' color)

In [59]:
with r.pipeline() as pipe:
    for hat in r.keys("hat:*"):
        c = r.hget(hat, "color")
        if c == b"black":
            pipe.delete(hat)
    pipe.execute()

#### Check if the 'black' hats are already deleted

In [53]:
for key in r.keys("hat*"):
    pprint(r.hgetall(key))
    print("\n")

{b'color': b'green',
 b'npurchased': b'200',
 b'price': b'99.99',
 b'quantity': b'0',
 b'style': b'baseball'}


{b'color': b'maroon',
 b'npurchased': b'7',
 b'price': b'59.99',
 b'quantity': b'493',
 b'style': b'hipster'}




#### Delete elements (Delete hats with quantity less than 500)

In [60]:
with r.pipeline() as pipe:
    for hat in r.keys("hat:*"):
        c = r.hget(hat, "quantity")
        if c < b"500":
            pipe.delete(hat)
    pipe.execute()

#### Check if the hats with quantity less than 500 are already deleted

In [61]:
for key in r.keys("hat*"):
    print(r.hgetall(key))

{b'color': b'maroon', b'price': b'59.99', b'style': b'hipster', b'quantity': b'500', b'npurchased': b'0'}


#### Update the values of 'hat:1236154736' , and show the remaining hats before and after this update
- By changing its color to '**brown**'
- and by adding '**updated**' flag/field to its hash values.

In [62]:
with r.pipeline() as pipe:
    pipe.multi()
    r.hset("hat:1236154736", "color", "brown")
    r.hset("hat:1236154736", "updated", "hat:1236154736")

## Task 4:

#### Create a simple Redis DB out of this relational model

- <b>Notes before starting this task:</b> 
- Redis Store is not invented for keeping relational data and posing structured queries, but for fast accesible data. 
- But just for keeping consistency with the our running example of how to represent the relational model in different NoSQL backends, we will try this out.


<b>Hints</b>:
* It’s quite straightforward to map your relational table into Redis data structures.
* **Hash**, **Sorted Set** and **Set** are the most useful data structures in this effort.
* This means that relationships are typically represented by **sets**.
* A set can be used to represent a **one-way relationship**, so you need one set per object to represent a **many-to-many** relationship.

#### The DataBase Model: 


This is  a toy DB about movies and actors who played roles in these movies. This DB is consisted of  

- A "Person" table which has a unique id, and a name fields.

- Another "Movie" table that has a unique id, a title, a country where it was made, and a year when it was released.

- There is (m-n) or "many-many" relationship between these two tables (i.e basically, many actors can act in many movies, and the movie include many actors)
- Therefore, we use the "Roles" table in which we can deduct which person has acted in which movie, and what role(s) they played.

<img src="figs/RDBSchema.png" alt="3" border="0">

* **Notice** that we will change the Redis DB to "**db(1)**"

    - By default there are <b>16</b> databases (indexed from 0 to 15), and you can navigate between them using <b>select</b> command.
    - Number of databases can be changed in the <b>redis config</b> file with databases setting.

    - By default, it selects the <b> database 0 </b>. 
    - To select a specified one, use <b> "redis-cli -n 1 " </b>  or use <b>   ("SELECT 1") </b> if you are already in the command line with one DB and wants to switch to DB 1. 

In [63]:
redis1= redis.Redis(host="localhost",port=6379, db=1)

### CREATE && INSERT in REDIS

#### 1. Creating a HashSet for the movies

In [64]:
#Helping function that insert multiple hashes of (key and value)into a Hashet
def setHash(mkey,mval):
    redis1.hmset(mkey,mval)

##### Movies

In [65]:
moviesLst=[
    
    ( "movie:1",{ 'title': 'Wall Street' , 'country':'USA', 'year':'1987' } ),
    ( "movie:2",{ 'title': 'The American President' , 'country':'USA', 'year':'1995' } ),
    ( "movie:3",{ 'title': 'Shawshank Redemption' , 'country':'USA', 'year':'1994' } )
]
for key, val in moviesLst:
    setHash(key, val)
#YOUR CODE LINES HERE to ADD this movie Hashes to REDIS.

#### Persons

In [66]:
personsLst=[
    
    ( "person:1",{ 'name': 'Charlie Sheen' } ),
    ( "person:2",{ 'name': 'Michael Douglas' } ),
    ( "person:3",{ 'name': 'Martin Sheen' } ),
    ( "person:4",{ 'name': 'Morgan Freeman' } )
]

#YOUR CODE LINES HERE to ADD this person Hashes to REDIS.
for key, val in personsLst:
    setHash(key, val)

#### Maintianig  the relationships in Redis

In [67]:
# Helping function that  create a set of a given values
def addSet(mkey,mvals):
    redis1.sadd(mkey,*mvals)
    

In [68]:
# Let's establish the many-to-many relationship 'roles'

# 1. For each movie, we keep a set of reference on the persons
movie_persons_list=[("movie:1:actors", ["person:1", "person:2", "person:3"] ),
                    ("movie:2:actors", ["person:2", "person:3" ]) ,
                    ("movie:3:actors", ["person:4" ])
                   ]
             
for (mkey,mvals) in movie_persons_list:
    addSet(mkey,mvals)
    
    
    
# 2. For each (person), we keep a set of reference on the (Movies)

person_movies_list=[("person:1:movies", ["movie:1"] ),
                    ("person:2:movies", ["movie:1", "movie:2" ]) ,
                    ("person:3:movies", ["movie:1", "movie:2" ]),
                    ("person:4:movies", ["movie:3" ])
                   ]

for (mkey,mvals) in person_movies_list:
    addSet(mkey,mvals)


# 3. For each (person_movie Acted_in relationship), We keep a set of roles made by the actor in each movie

persons_movies_roles_lists = [("person:1:movie:1", ["Bud Fox"]),
                              ("person:2:movie:1", ["Gordon Gekko"]),
                              ("person:3:movie:1", ["Carl Fox"]),
                              ("person:2:movie:2", ["President Andrew Shepherd"]),
                              ("person:3:movie:2", ["A.J. McInerney"]),
                              ("person:4:movie:3", ["Ellis Boyd"]),
]

for (mkey,mvals) in persons_movies_roles_lists:
    addSet(mkey,mvals)

#ADD YOUR CODE LINES HERE 

### Querying our REDIS Data

#### <font color= 'red'>Important to remeber:</font>
- The only way to query Redis is using the **"Keys"**, so we mainly depend on the programming cabailities of python, and the datascience cabailities of **Pandas** to make the data science part :)
- One of the Cons of Redis is that youy can't query on **values**!!

#### Get Persons in your Redis DB

In [85]:
redis1.keys("person:?")

[b'person:4', b'person:1', b'person:3', b'person:2']

#### We can use pandas for showing the results in a better way !

In [86]:
data=[]
for key in redis1.keys("person:?"):
    name = redis1.hvals(key)[0].decode()
    data. append({'name': name})
    
df=pd.DataFrame(data)
display(df)

,name
0,Morgan Freeman
1,Charlie Sheen
2,Martin Sheen
3,Michael Douglas


#### Get persons with names start with 'C' letter

In [91]:
for p in redis1.keys("person:?"):
    if redis1.hget(p, "name").startswith(b"C"):
        print(redis1.hget(p, "name"))

b'Charlie Sheen'


#### Use Pandas to show the result in a better way!

In [93]:
data=[]
for key in redis1.keys("person:?"):
    name = redis1.hvals(key)[0].decode()
    if name.startswith("C"):
        data. append({'name': name})
    
df=pd.DataFrame(data)
display(df)

,name
0,Charlie Sheen


#### Get All Movies , sorted from recent to old

In [101]:
data=[]
for key in redis1.keys("movie:?"):
    d = redis1.hgetall(key)
    data.append({'title': d[b'title'].decode(), 
                  'country': d[b'country'].decode(),
                  'year': d[b"year"].decode()})
    
df=pd.DataFrame(data)
df.sort_values(by ="year", inplace=True, ascending=True)
display(df)

,title,country,year
0,Wall Street,USA,1987
2,Shawshank Redemption,USA,1994
1,The American President,USA,1995


#### Get All Movies released in the 90s (after year (1990) and before 2000)

In [103]:


data = []
for key in redis1.keys("movie:?"):  # safer than .keys()
    d = redis1.hgetall(key)
    
    # decode fields safely
    title = d.get(b'title', b'').decode()
    country = d.get(b'country', b'').decode()
    year = int(d.get(b'year', b'0').decode() or 0)
    
    # filter only 1990s (exclusive of 1990, before 2000)
    if 1990 < year < 2000:
        data.append({
            'title': title,
            'country': country,
            'year': year
        })

# Convert to DataFrame
df = pd.DataFrame(data)

# Sort by year ascending
df.sort_values(by="year", inplace=True, ascending=True)

display(df)


,title,country,year
1,Shawshank Redemption,USA,1994
0,The American President,USA,1995


In [ ]:
#YOUR CODE HERE

### Querying from multiple tables

#### Get Movies and Actors from the DB


In [104]:
for key in redis1.keys("person:?:movies"):
    # personKey deserliaze the key get 'person:1', 'person:2',...
    personKey=key.decode().split(':')[0]+ ":" +key.decode().split(':')[1]
    # get the persons from the hashsets using the above key
    person=redis1.hgetall(personKey)
    # SMEMBERS  get the movies of each person using the 'key' in the beginning
    personMoviesKeys=redis1.smembers(key)
    for movKey in personMoviesKeys:
        print(person.get(b'name').decode(),":   ", redis1.hgetall(movKey).get(b'title').decode())  

Michael Douglas :    Wall Street
Michael Douglas :    The American President
Charlie Sheen :    Wall Street
Martin Sheen :    Wall Street
Martin Sheen :    The American President
Morgan Freeman :    Shawshank Redemption


In [106]:

data = []

# Iterate through all person:<id>:movies sets
for key in redis1.scan_iter("person:*:movies"):  # safer than keys()
    parts = key.decode().split(':')
    personKey = f"{parts[0]}:{parts[1]}"  # e.g., person:1
    
    # Get person info
    person = redis1.hgetall(personKey)
    person_name = person.get(b'name', b'').decode()
    person_age = person.get(b'age', b'').decode() if b'age' in person else None
    
    # Get all movie keys for this person
    personMoviesKeys = redis1.smembers(key)
    
    # For each movie, get its details
    for movKey in personMoviesKeys:
        movie = redis1.hgetall(movKey)
        if movie:
            movie_title = movie.get(b'title', b'').decode()
            movie_year = movie.get(b'year', b'').decode()
            
            data.append({
                'person': person_name,
                'age': person_age,
                'movie_title': movie_title,
                'movie_year': movie_year
            })

# Create a DataFrame
df = pd.DataFrame(data)

# Optional: sort by person then year
df.sort_values(by=['person', 'movie_year'], inplace=True)

display(df)


,person,age,movie_title,movie_year
2,Charlie Sheen,None,Wall Street,1987
3,Martin Sheen,None,Wall Street,1987
4,Martin Sheen,None,The American President,1995
0,Michael Douglas,None,Wall Street,1987
1,Michael Douglas,None,The American President,1995
5,Morgan Freeman,None,Shawshank Redemption,1994


#### Get count of "Movies" in your DB

- Note: The best way is to store the sum as a separate key, and to update whenever you add/remove a value from your set/hash/zset.

In [108]:
count = len(list(redis1.keys("movie:?")))
print("Total movies:", count)


Total movies: 3


#### In this DB, for every "Actor" get the number of movies he played

In [ ]:
#YOUR CODE HERE

In [ ]:
## or using Pnadas DFs
#YOUR CODE HERE    

#### In this DB, List the movies that every Actor Played

In [ ]:
#YOUR CODE HERE

In [ ]:
##or using Pandas,and SQL
import pandasql as ps
import re 

#YOUR CODE HERE

### Updating Redis Data
* If the key or hash field already exists in the hash, they are overwritten.
- update the year the wallstreet movie was released in to be 2000, which is not true BTW :)
- Show that movie before and After updating it

In [ ]:
#YOUR CODE HERE

####  Delete all the persons with names start with 'M' letter.

In [ ]:
#YOUR CODE HERE

 ## How long did it take you to solve the homework?
 
Please answer as precisely as you can. It does not affect your points or grade in any way. It is okey, if it took 0.5 hours or 24 hours. The collected information will be used to improve future homeworks.

<font color="red"><b>Answer:</b></font>

**<center> <font color='red'>THANK YOU FOR YOUR EFFORT!</font></center>**